In [ ]:
r"""
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Reduced order model for a thermal system of electronic circuit developed via Lumped Parameter Linear Superposition (LPLSP) method. 
This code maps the heat generated at source (Watt) within the system to the temperature reponse (degree C) at any location within the system. 
It works seamlessly for a system with many sources and many monitor points, which may be present on the source or away from the source.

This model works in two stages:
1. Training - In this stage, the algorithm learns various parameters that physically relate the heat dissipated at the source (input) to the temperature response at a monitor point (output). 
    1.1. This model reduces a thermal system to lumped parameters (thermal resistance [R] - physically represents a component's resistance to heat flow, thermal capacitance [C] - physically represents a component's ability to store heat, time constant [K] = 1/RC)
    1.2. It relates the heat loss in an electronic component (input power dissipation in Watts) to spatial temperature change (for example temperature within the component or on the case or at a location on the PCBA in degree C) using equation
                T_i(t) = T_0 + \sum_{j=1} ^{N} \sum_{m} \Delta(P_j(t)*R_ij)_{m} *(1-e^{-K_ij*(t-t_m)})
                where, j is the number of heat sources and m is the index of change of input
    1.3. The terms R_ij (thermal resistance matrix) and K_ij (time constant matrix) generally require several sequential simulations, where a unit input power (1W) is applied to each source and the temperature response of all the monitor points are measured. Then a superposition matrix is created.
    1.4. However, in this case instead of sequential simulations, a single transient simulation is run where all the heat sources are excited at the same time and all the temperature monitor points are measured. Then a parameter matrix for R,K that map output T to input P are estimated simultaneously.
    1.5. Once estimated, the parameter matrices R,K are saved.
    
2. Testing / Prediction - In this stage, the algorithm uses the saved parameters (representation of the full system) to predict temperature response for new input power dissipations. The prediction script
is similar to the one in LPLSP-EPE_Rank_Reduced.ipynb file.

This approach is very efficient because it reduces not only the the time required to predict temperature of a thermal system for new unseen inputs, but also reduces the time required to create the reduced order model which is a major bottleneck for large systems.

Complete description of this approach is presented in the following papers:
a. Background of the method, derivation and case studies 
    * N. Padmanabhan, "A Transient Thermal Model for Power Electronics Systems," SoutheastCon 2024, Atlanta, GA, USA, 2024, pp. 1294-1299, doi: 10.1109/SoutheastCon52093.2024.10500091.
    * Padmanabhan, N., "A Scalable Approach for Transient Thermal Modeling of Automotive Power Electronics," SAE Technical Paper Series, January 1, 2025, https://doi.org/10.4271/2025-01-5073.

b. Ensemble parameter estimation method, algorithm and case studies
    * Padmanabhan, N., "Rapid Modeling of Transient Thermal Systems through Ensemble Parameter Estimation Applied to the Lumped Parameter Linear Superposition Method," SAE Technical Paper Series, January 1, 2026, https://doi.org/10.4271/2026-01-5034.

This approach is generally applicable to any thermal system where conduction and convection modes dominate the heat transfer. However, extension to other systems will require careful review of the system (sources and sinks), units of heat generation, temperature response
and the dominant modes of heat transfer. While this approach can be expanded to non-linear convection (convection with variable flow velocities) and radiation, this code as is does not model these cases accurately. A separate code for the non-linear cases
will be posted.

SUMMARY OF THE CODE: 
This script identifies thermal coupling parameters between multiple power sources and multiple temperature sensors.

Model:
    Temperature = Sum of first-order thermal responses to input power changes.

Parameters estimated:
    R : thermal resistance matrix
    K : time constant matrix

Optimizer:
    scipy.optimize.least_squares()

Input file:
    Training.xlsx

    Format: Training data arranged in columns labeled t, P_j, T_i

Output file:
    parameters_rank_reduced.xlsx
"""
# STAGE 1 - PARAMETER ESTIMATION 

import numpy as np
from numba import njit
import pandas as pd
from scipy.optimize import least_squares
import time

#-------------------------------------------------------------------------------------------------------------------
# Predict temperature for one monitor point 
@njit(cache=True, fastmath=True)
def temperature_model(P, t, R_row, K_row, T0):
    """
    Parameters:
    P: ndarray(n_inputs, t_len) - Input power dissipation (W)
    t: ndarray(t_len,) - Time vector (s)
    R_row: ndarray(n_inputs,) - Thermal resistance coupling coefficients between each source and monitor points
    K_row: ndarray(n_inputs,) - Time constant coupling coefficients between each source and monitor points
    T_0: float - Ambient or reference temperature

    Output:
    Predicted temperature profile - T_0 + T_out: ndarray(t_len,)
    """
    n_inputs, t_len = P.shape
    T_out = np.zeros(t_len)

    # Process contribution from each power source 
    for j in range(n_inputs):
        Rj = R_row[j]
        Kj = K_row[j]

        # Create a local copy because we modify the first sample. The first sample is forced to zero as the initial value is not interpreted as a temperature step. 
        Pj = P[j].copy()
        Pj[0] = 0.0

        # Detect indices where the input power changes. It is at these indices where add a new summation of delta T. If the power is constant, the temperature curve has a first order response, if it varies then we have a summation of several
        # first order responses.
        idx_list = []
        for k in range(1, t_len):
            if Pj[k] != Pj[k-1]:
                idx_list.append(k)
        
        if len(idx_list) == 0:
            continue

        # Convert power to temperature response
        TC = Pj * Rj

        # Add the temperature response from every detected index change
        for idx in idx_list:
            delta = TC[idx] - TC[idx - 1]   # Magnitude of step 
            t0 = t[idx - 1]                 # time index where the input changes

            dt = t[idx:t_len] - t0          # time elapsed since index change occured

            # First order response - LPLSP. Applied to all future time points at the same time (vectorized computation)
            T_out[idx:t_len] += delta * (1 - np.exp(-Kj * dt))

    return T0 + T_out
#-------------------------------------------------------------------------------------------------------------------
# Residual Function 
# Compute sum of square error between temperature measured from the simulation (T_meas) and temperature predicted by the model (T_pred)
# and find the parameters that minimize the SSE.
#
# theta ->  build R,K matrices -> predict T_pred -> compare with T_meas -> compute SSE -> minimize SSE
# over all temperatures and time points.

def residuals(theta, t, P, T_meas, T0):
    N = P.shape[0]
    NN = N*N

    R = theta[:NN].reshape(N, N)
    K = theta[NN:].reshape(N, N)

    T_pred = np.zeros_like(T_meas)

    for i in range(N):
        T_pred[i] = temperature_model(P, t, R[i], K[i], T0)

    return (T_pred - T_meas).ravel()


# MAIN TRAINING SCRIPT FOR PARAMETER ESTIMATION

# 1. Load training data
# 2. Build power and temperature arrays
# 3. Define optimization settings
# 4. Estimate R and K matrices
# 5. Save identified parameters

if __name__ == "__main__":
    
    df = pd.read_excel("Training.xlsx")

    # Automatic identification of time vector, temperature columns, power columns. It is recommended the training file Training.xlsx has headers t, T_1, T_2, T_3 ..., P_1, P_2, P_3 etc.
    t = df["t"].values
    P_cols = [col for col in df.columns if col.startswith("P")]
    T_cols = [col for col in df.columns if col.startswith("T")]
    
    # Number of sources
    N = len(P_cols)
    
    # Power input array P : shape (N, T)
    P = np.vstack([df[col].values for col in P_cols])
    
    # Measured temperatures T_meas : shape (N, T)
    T_meas = np.vstack([df[col].values for col in T_cols])
    
    T0 = 20.0   # Initial temperature

    # Initial guess of 1 is used for all parameters 
    theta0 = np.ones(2 * N * N)
    
    
    # OPTIMIZATION
    
    print("\n Start Optimization")
    # Measure computation time
    start = time.time()

    # Methods - trf (trust reflective  / dogbox / lm - While all methods work, trf is observed to be most effective and allows usage of bounds. 
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html
    
    result = least_squares(
        residuals,
        theta0,
        args=(t, P, T_meas, T0),
        method="trf",      # recommended after your tests
        max_nfev=10000,
        verbose=2
    )
    
    theta_est = result.x
    
    print("\n Optimization complete in %.2f s" % (time.time() - start))
    
    # EXTRACT FINAL MATRICES
    
    NN = N * N
    R_est = theta_est[:NN].reshape(N, N)
    K_est = theta_est[NN:].reshape(N, N)
    
    for i in range(N):
        for j in range(N):
            print(f"R{i+1}{j+1} = {R_est[i,j]:.6f};")
        print()
    
    for i in range(N):
        for j in range(N):
            print(f"K{i+1}{j+1} = {K_est[i,j]:.6f};")
        print()
    
    
    # SAVE PARAMETERS
    
    out_names = []
    out_values = []
    
    # Flatten R
    for i in range(N):
        for j in range(N):
            out_names.append(f"R{i+1}{j+1}")
            out_values.append(R_est[i, j])
    
    # Flatten K
    for i in range(N):
        for j in range(N):
            out_names.append(f"K{i+1}{j+1}")
            out_values.append(K_est[i, j])
    
    # Build DataFrame
    df_out = pd.DataFrame({
        "Parameter": out_names,
        "Value": out_values
    })
    
    # Write to Excel
    df_out.to_excel("parameters_naive.xlsx", index=False)

